In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


## 0. Read Data for Analysis

In [ ]:
df = pd.read_csv('data/hotel_bookings.csv')

In [ ]:
df.dtypes

In [ ]:
df.columns

## 1. Data Cleaning


In [ ]:
filter1 = (df['children'] == 0) & (df['adults'] == 0) & (df['babies'] == 0)

In [ ]:
df[filter1]

In [ ]:
df2 = df[~filter1]

In [ ]:
df2.duplicated().sum()

In [ ]:
data = df2.drop_duplicates().copy()

In [ ]:
data.shape

## 2. EDA

## 2.1 Descriptive Analysis

In [ ]:
data[['lead_time', 'total_of_special_requests', 'adr']].describe().T

In [ ]:
data.info(memory_usage='deep')

In [ ]:
for col in ['lead_time', 'total_of_special_requests', 'adr']:
    print('feature name : {}'.format(col) )
    
    for i in range (90,101,1):
        quantile_value = np.quantile(data[col], q = i/100)
        print('{}th quantile value is {}'.format(i,quantile_value))
    print('\n')


#### Upper-Tail Analysis

The 90th to 100th percentiles are examined to identify unusually high values and potential outliers in lead time, special requests, and ADR.

## 2.2 Where do the bookings come from

In [ ]:
not_cancelled = data[data['is_canceled'] == 0]

In [ ]:
country_wise_data = not_cancelled['country'].value_counts().reset_index()

In [ ]:
country_wise_data.columns = ['country', 'no_of_bookings']

In [ ]:
country_wise_data

In [ ]:
import chart_studio.plotly as py
import plotly.graph_objs as go
import plotly.express as px
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

init_notebook_mode(connected=True)

In [ ]:
map_guest = px.choropleth(data_frame=country_wise_data,
              locations = country_wise_data['country'],
              color = country_wise_data['no_of_bookings'],
              hover_name= country_wise_data['country'],
              title = 'Non-Cancelled Hotel Bookings by Country'
              )

In [ ]:
map_guest.show()

In [ ]:
top_countries = country_wise_data.head(10)

In [ ]:
top_countries

In [ ]:
country_wise_data['guest_percentage'] = (
    country_wise_data['no_of_bookings']
    / country_wise_data['no_of_bookings'].sum()
    * 100
)

country_wise_data.head(10)

In [ ]:
top_country = country_wise_data.iloc[0]

print(
    f"The largest number of non-cancelled bookings comes from "
    f"{top_country['country']}, accounting for "
    f"{top_country['guest_percentage']:.2f}% of non-cancelled bookings."
)

## 2.3 How often does assigned room type match reserved room type

In [ ]:
data.columns

In [ ]:
pivot = pd.crosstab(index = data['reserved_room_type'], columns = data['assigned_room_type'], margins = True)

In [ ]:
pivot

In [ ]:
pivot_normalize = pd.crosstab(index = data['reserved_room_type'], columns = data['assigned_room_type'], margins = True, normalize = 'index').round(2).mul(100)

In [ ]:
pivot_normalize

In [ ]:
room_match_pct = (
    data['reserved_room_type']
    .eq(data['assigned_room_type'])
    .mean()
    * 100
)

print(
    f'{room_match_pct:.2f}% of bookings were assigned '
    f'the same room type that was reserved.'
)

In [ ]:
room_mismatch_pct = 100 - room_match_pct

print(
    f'{room_mismatch_pct:.2f}% of bookings were assigned '
    f'a different room type.'
)

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    pivot_normalize.drop(
        index='All',
        columns='All',
        errors='ignore'
    ),
    annot=True,
    fmt='.0f',
    cmap='Blues'
)

plt.title('Reserved vs Assigned Room Types')
plt.xlabel('Assigned Room Type')
plt.ylabel('Reserved Room Type')
plt.show()

## 2.4 Which market segment generates the most bookings

In [ ]:
data['market_segment'].value_counts()

In [ ]:
fig = px.pie(
    values=data['market_segment'].value_counts().values,
    names=data['market_segment'].value_counts().index,
    title='Distribution of Bookings by Market Segment'
)

In [ ]:
fig.show()

In [ ]:
market_segment_pct = (
    data['market_segment']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

market_segment_pct

In [ ]:
largest_segment = market_segment_pct.idxmax()
largest_segment_pct = market_segment_pct.max()

print(
    f'{largest_segment} is the largest market segment, '
    f'accounting for {largest_segment_pct:.2f}% of bookings.'
)

## 2.5 How does ADR vary by market segment and room type

In [ ]:
plt.figure(figsize=(20,10))

sns.barplot(
    x='market_segment',
    y='adr',
    hue='reserved_room_type',
    data=data
)

plt.title('Average Daily Rate by Market Segment and Reserved Room Type')
plt.xlabel('Market Segment')
plt.ylabel('Average Daily Rate (ADR)')
plt.legend(title='Reserved Room Type')
plt.show()

In [ ]:
adr_summary = (
    data
    .groupby(['market_segment', 'reserved_room_type'])['adr']
    .agg(
        number_of_bookings='count',
        average_adr='mean',
        median_adr='median'
    )
    .round(2)
)

adr_summary

## 2.6 How does guest arrival volume change over time

In [ ]:
data.head(1)

In [ ]:
data['arrival_date_month'].unique()

In [ ]:
dict_month = {
    'July' : 7,    'August' : 8, 'September' : 9,   'October' : 10,  'November' : 11,  'December' : 12,
   'January' : 1,  'February' : 2,     'March' : 3,     'April' : 4,       'May' : 5,      'June' : 6
}

In [ ]:
data['arrival_date_month_index'] = data['arrival_date_month'].map(dict_month)

In [ ]:
data.columns

In [ ]:
data['arrival_date'] = pd.to_datetime(
    dict(
        year=data['arrival_date_year'],
        month=data['arrival_date_month_index'],
        day=data['arrival_date_day_of_month']
    )
)

In [ ]:
data['total_guests'] = (data['adults'] + data['children'] + data['babies'])

In [ ]:
data_no_cancel = data[
    data['is_canceled'] == 0
].copy()

In [ ]:
guest_arrival_series = (
    data_no_cancel
    .groupby('arrival_date')['total_guests']
    .sum()
    .sort_index()
)

In [ ]:
data.head(2)

In [ ]:
guest_arrival_series.plot(figsize=(12, 6))

plt.title('Total Guest Arrivals Over Time')
plt.xlabel('Arrival Date')
plt.ylabel('Number of Guests')
plt.show()

In [ ]:
busiest_date = guest_arrival_series.idxmax()
max_guests = guest_arrival_series.max()

print(
    f'The busiest arrival date was '
    f'{busiest_date:%d %B %Y}, '
    f'with {max_guests:.0f} guests.'
)

In [ ]:
quietest_date = guest_arrival_series.idxmin()
min_guests = guest_arrival_series.min()

print(
    f'The quietest arrival date was '
    f'{quietest_date:%d %B %Y}, '
    f'with {min_guests:.0f} guests.'
)

## 2.7 Analyzing distribution of daily guest arrival

In [ ]:
guest_arrival_series.values

In [ ]:
g = sns.displot(
    guest_arrival_series.values,
    bins=30
)

g.set_axis_labels(
    'Number of Guests per Day',
    'Frequency'
)

g.fig.suptitle(
    'Distribution of Daily Guest Arrivals',
    y=1.03
)

plt.show()


In [ ]:
np.mean(guest_arrival_series.values)

In [ ]:
np.median(guest_arrival_series.values)

In [ ]:
np.std(guest_arrival_series.values)

In [ ]:
arrival_summary = pd.Series({
'mean_daily_guests': guest_arrival_series.mean(),
'median_daily_guests': guest_arrival_series.median(),
'std_daily_guests': guest_arrival_series.std(),
'minimum_daily_guests': guest_arrival_series.min(),
'maximum_daily_guests': guest_arrival_series.max()
}).round(2)
arrival_summary

## 2.8 Which market segments have the highest cancellation rates

In [ ]:
cancellation_by_segment = (
    data
    .groupby('market_segment')['is_canceled']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
)

cancellation_by_segment

In [ ]:
cancellation_by_segment.plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title('Cancellation Rate by Market Segment')
plt.xlabel('Market Segment')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=45)
plt.show()